# Dataset Transliteration Pipeline
Converts native-script datasets (Hindi, Bengali, etc.) → Romanized versions (Hinglish, Banglish, etc.)

**Structure:**
-  Cell 1 — Install dependencies
-  Cell 2 — **VARIABLE PART** ← change this per dataset
-  Cell 3 — Fixed: loader
-  Cell 4 — Fixed: transliterator
-  Cell 5 — Fixed: normalizer + validator
-  Cell 6 — Fixed: run pipeline & save
-  Cell 7 — Fixed: merge all outputs

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install indic-transliteration pandas tqdm -q

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙️  VARIABLE PART  —  Edit this cell only when switching datasets
# ══════════════════════════════════════════════════════════════════════════════

# ── File ──────────────────────────────────────────────────────────────────────
INPUT_PATH   = "data/hindi_dataset.csv"   # path to your raw dataset
OUTPUT_PATH  = "output/hindi_out.tsv"     # where to write the cleaned output
FILE_FORMAT  = "csv"                      # 'csv' | 'tsv' | 'json' | 'excel'

# ── Column names in the raw file ──────────────────────────────────────────────
TEXT_COL     = "text"                     # column that holds the text
LABEL_COL    = "label"                    # column that holds the class label

# ── Label mapping  →  must resolve to 0 or 1 ─────────────────────────────────
# Examples:
#   {"hate": 1, "not_hate": 0}
#   {"offensive": 1, "non-offensive": 0}
#   {1: 1, 0: 0}
LABEL_MAP    = {"hate": 1, "not_hate": 0}

# ── Language / transliteration ────────────────────────────────────────────────
LANGUAGE_TAG = "hinglish"                 # tag written into output (free text)

# Script of the INPUT text — pick one:
#   "devanagari"   → Hindi, Marathi, Nepali …
#   "bengali"      → Bengali / Bangla
#   "gurmukhi"     → Punjabi
#   "gujarati"     → Gujarati
#   "telugu"       → Telugu
#   "kannada"      → Kannada
#   "malayalam"    → Malayalam
#   "tamil"        → Tamil
#   "oriya"        → Odia
#   "roman"        → already Roman script (skip transliteration)
INPUT_SCRIPT = "devanagari"

# Transliteration scheme for OUTPUT — pick one:
#   "itrans"       → casual Hinglish feel  (namaste, pyaar)
#   "iast"         → academic Roman        (namaste, pyāra)
#   "hk"           → Harvard-Kyoto
#   "kolkata"      → National Library Calcutta
OUTPUT_SCHEME = "itrans"

# ── Extra read_csv / read_json kwargs (optional) ─────────────────────────────
READ_KWARGS  = {}                         # e.g. {"encoding": "utf-8", "sep": "|"}

In [ ]:
# ── Cell 3 (FIXED): Imports & loader ──────────────────────────────────────────
import re, os, glob
import pandas as pd
from tqdm import tqdm
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ── Script map ────────────────────────────────────────────────────────────────
SCRIPT_MAP = {
    "devanagari" : sanscript.DEVANAGARI,
    "bengali"    : sanscript.BENGALI,
    "gurmukhi"   : sanscript.GURMUKHI,
    "gujarati"   : sanscript.GUJARATI,
    "telugu"     : sanscript.TELUGU,
    "kannada"    : sanscript.KANNADA,
    "malayalam"  : sanscript.MALAYALAM,
    "tamil"      : sanscript.TAMIL,
    "oriya"      : sanscript.ORIYA,
    "roman"      : None,
}
SCHEME_MAP = {
    "itrans"   : sanscript.ITRANS,
    "iast"     : sanscript.IAST,
    "hk"       : sanscript.HK,
    "kolkata"  : sanscript.KOLKATA,
}

SRC_SCRIPT  = SCRIPT_MAP.get(INPUT_SCRIPT)
TGT_SCHEME  = SCHEME_MAP.get(OUTPUT_SCHEME, sanscript.ITRANS)

def load_dataset(path, fmt, kwargs):
    fmt = fmt.lower()
    if fmt == "csv":
        return pd.read_csv(path, **kwargs)
    elif fmt == "tsv":
        return pd.read_csv(path, sep="\t", **kwargs)
    elif fmt == "json":
        return pd.read_json(path, **kwargs)
    elif fmt in ("excel", "xlsx", "xls"):
        return pd.read_excel(path, **kwargs)
    else:
        raise ValueError(f"Unsupported format: {fmt}")

df = load_dataset(INPUT_PATH, FILE_FORMAT, READ_KWARGS)
print(f"✅ Loaded {len(df):,} rows from '{INPUT_PATH}'")
print(f"   Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# ── Cell 4 (FIXED): Transliterator ────────────────────────────────────────────

def transliterate_text(text: str) -> str:
    """Convert native-script text to Roman. Skips if already Roman."""
    if SRC_SCRIPT is None:          # input is already Roman
        return text
    try:
        return transliterate(text, SRC_SCRIPT, TGT_SCHEME)
    except Exception:
        return text                 # fallback: keep original

In [ ]:
# ── Cell 5 (FIXED): Normalize + validate ──────────────────────────────────────

def normalize(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)          # collapse whitespace
    text = re.sub(r'[^\w\s.,!?\'-]', '', text) # strip stray special chars
    text = text.lower()
    return text

def map_label(raw_label):
    """Map raw label to 0/1 using LABEL_MAP. Returns None if unknown."""
    return LABEL_MAP.get(raw_label, None)

def is_valid(text: str, label) -> bool:
    if not text or len(text) < 3:  return False
    if label not in (0, 1):        return False
    return True

In [ ]:
# ── Cell 6 (FIXED): Run pipeline & save ───────────────────────────────────────

rows   = []
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    raw_text  = row.get(TEXT_COL, "")
    raw_label = row.get(LABEL_COL)

    # 1. Transliterate
    roman_text = transliterate_text(str(raw_text))

    # 2. Normalize
    clean_text = normalize(roman_text)

    # 3. Map label
    label = map_label(raw_label)

    # 4. Validate
    if not is_valid(clean_text, label):
        skipped += 1
        continue

    rows.append({
        "text"      : clean_text,
        "offensive" : label,
        "language"  : LANGUAGE_TAG,
    })

out = pd.DataFrame(rows)
out.to_csv(OUTPUT_PATH, sep="\t", index=False)

print(f"\n✅ Saved  → {OUTPUT_PATH}")
print(f"   Kept   : {len(out):,} rows")
print(f"   Skipped: {skipped:,} rows")
print(f"\nLabel distribution:")
print(out["offensive"].value_counts().to_string())
print("\nSample output:")
out.head()

In [ ]:
# ── Cell 7 (FIXED): Merge all *_out.tsv files into final_dataset.tsv ──────────
# Run this AFTER processing every dataset through the pipeline above.

tsv_files = glob.glob("output/*_out.tsv")

if not tsv_files:
    print("⚠️  No output TSV files found. Run the pipeline for at least one dataset first.")
else:
    dfs = [pd.read_csv(f, sep="\t") for f in tsv_files]
    final = pd.concat(dfs, ignore_index=True)
    final = final.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

    print("── Merged from ──────────────────────────")
    for f in tsv_files:
        print(f"   {f}")

    print(f"\nTotal rows : {len(final):,}")
    print("\nBy language:")
    print(final["language"].value_counts().to_string())
    print("\nBy label:")
    print(final["offensive"].value_counts().to_string())

    final.to_csv("output/final_dataset.tsv", sep="\t", index=False)
    print("\n✅ Saved → output/final_dataset.tsv")
    final.head()